# Pregunta 1 — Entrenamiento de los modelos A y B en Colab

Requisitos: `colab_bundle.zip` (generado con `python src/s07_pack_colab.py`) subido a
`MyDrive/knee/` en una carpeta **privada** (datos derivados de Kaggle).

Entorno: *Entorno de ejecución → Cambiar tipo → GPU* (L4 o A100 en Colab Pro).
Cada corrida guarda `last.pt` en Drive al final de cada época: si Colab se desconecta,
basta volver a ejecutar la celda de corridas (usa `--resume`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/knee'
WORK = '/content/knee'
RUNS_DIR = f'{DRIVE}/runs_v2'      # v1 (runs/) queda como primer intento
REPORTS_DIR = f'{DRIVE}/reports_v2'
# Copia al disco local de Colab: leer desde Drive archivo por archivo es lento
!mkdir -p {WORK} && cp {DRIVE}/colab_bundle.zip /content/ && unzip -q -o /content/colab_bundle.zip -d {WORK}
# Código actualizado (si existe): pisa src/ del paquete grande
import os
if os.path.exists(f'{DRIVE}/colab_code.zip'):
    !unzip -q -o {DRIVE}/colab_code.zip -d {WORK}
%cd {WORK}/src
!ls ../data/processed

## Corridas

| Nombre | Qué cambia frente a `B_base` | Entregable |
|---|---|---|
| `A_base` | CNN desde cero (modelo A) | 2 |
| `B_base` | ResNet-50 ImageNet, sagital, BCE + pos_weight, etiquetas duras | 2, 3 |
| `B_focal` | pérdida focal (γ = 2) | 4 (pérdida) |
| `B_cor` | plano coronal | 4 (plano) |
| `B_conf` | pondera cada etiqueta por su confianza | 4 (ruido) |
| `B_hc` | solo etiquetas de alta confianza | 4 (ruido) |
| `B_nopw` | sin `pos_weight` (sin mitigación del desbalance) | 6 |
| `B_all` | reentrena toda la ResNet (no solo layer3/4) | 4 (profundidad) |
| `B_seed1`, `B_seed2` | otra semilla: mide la variabilidad por azar | rigor |

Para repartir entre integrantes, cada uno deja en `RUN` solo sus corridas.

In [ ]:
RUNS = {
    'A_base':   '--arch scratch',
    'B_base':   '--arch resnet50',
    # ablaciones (E4) y desbalance (E6): cambian UN factor frente a B_base
    'B_focal':  '--arch resnet50 --loss focal',
    'B_cor':    '--arch resnet50 --view cor',
    'B_conf':   '--arch resnet50 --noise conf',
    'B_hc':     '--arch resnet50 --noise hc',
    'B_nopw':   '--arch resnet50 --no-pos-weight',
    'B_all':    '--arch resnet50 --unfreeze all',      # profundidad reentrenada
    # variabilidad entre semillas: cuánto cambia B_base solo por el azar
    'B_seed1':  '--arch resnet50 --seed 1',
    'B_seed2':  '--arch resnet50 --seed 2',
}
RUN = ['B_focal', 'B_cor', 'B_conf', 'B_hc', 'B_nopw', 'B_all', 'B_seed1', 'B_seed2']

for name in RUN:
    print(f'
==================== {name}')
    !python train.py {RUNS[name]} --workers 4 --resume --out {RUNS_DIR}/{name}

In [ ]:
import pandas as pd, glob, os
rows = []
for d in sorted(glob.glob(f'{RUNS_DIR}/*/history.csv')):
    h = pd.read_csv(d)
    rows.append(dict(run=os.path.basename(os.path.dirname(d)), epochs=len(h),
                     best_auc_weak_val=h.val_auc_weak.max(),
                     last_auc_train=h.train_auc_weak.iloc[-1] if 'train_auc_weak' in h else None, min_per_epoch=h.sec.mean() / 60))
pd.DataFrame(rows).round(3)

## Evaluación en los 58 dorados

AUC macro con IC por *bootstrap*, AUC por etiqueta, P/R/F1 al umbral elegido en `weak_val` y diferencia de cada ablación frente a `B_base` (IC pareado). Se puede correr con las corridas que ya terminaron.

In [ ]:
!python evaluate.py --runs {RUNS_DIR} --ref B_base
!mkdir -p {REPORTS_DIR} && cp ../reports/eval_* {REPORTS_DIR}/

## Grad-CAM (entregable 5)

Para cada hallazgo: los 3 dorados positivos con mayor probabilidad y el falso positivo más seguro, en el corte que la atención del modelo eligió. Además, la posición del corte más atendido en los 58 dorados (¿meniscos en los extremos y LCA al centro?). Las figuras quedan en `MyDrive/knee/reports_v2/gradcam/`.

In [ ]:
for name in ['B_base', 'A_base']:
    !python gradcam.py --run {RUNS_DIR}/{name}
!mkdir -p {REPORTS_DIR}/gradcam && cp -r ../reports/gradcam/* {REPORTS_DIR}/gradcam/

from IPython.display import Image, display
display(Image(f'{REPORTS_DIR}/gradcam/B_base/gradcam_panel.png'))